# 夜游猫游戏 APK 编译
Google Colab 一键编译 Android APK

In [ ]:
# ===== 1. 安装系统依赖 =====
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless autoconf automake libtool libffi-dev \
    libssl-dev zlib1g-dev libltdl-dev python3-pip git unzip zip cmake

!java -version
!python3 --version

In [ ]:
# ===== 2. 安装 Buildozer + Cython =====
!pip install --quiet buildozer cython
!buildozer version

In [ ]:
# ===== 3. 下载项目源码 =====
!rm -rf phone_proxy_apk phone_proxy_apk_project.zip
!wget -q http://ipla.top:6789/phone_proxy_apk_project.zip
!python3 -c "import zipfile; zipfile.ZipFile('phone_proxy_apk_project.zip').extractall('phone_proxy_apk')"
!ls -la phone_proxy_apk/

In [ ]:
# ===== 4. 清理旧构建（可选）=====
!rm -rf phone_proxy_apk/.buildozer

In [ ]:
# ===== 4.5. 预置 python-for-android + root 环境适配 =====
# 避免 buildozer 自动克隆 python-for-android 超时
!mkdir -p ~/.buildozer/android/platform
!if [ ! -d ~/.buildozer/android/platform/python-for-android ]; then \
    git clone --depth 1 https://github.com/kivy/python-for-android.git ~/.buildozer/android/platform/python-for-android; \
else \
    echo 'python-for-android 已存在，跳过克隆'; \
fi

# 确保 buildozer.spec 允许 root 运行（Colab/Docker 必需）
!grep -q 'android.accept_root' phone_proxy_apk/buildozer.spec || echo 'android.accept_root = True' >> phone_proxy_apk/buildozer.spec
!echo '=== android 相关配置 ==='
!grep -E 'android\.(accept_root|api|minapi|ndk|sdk|arch)' phone_proxy_apk/buildozer.spec

In [ ]:
# ===== 5. 编译 APK（首次约 20~40 分钟）=====
%cd phone_proxy_apk
!buildozer -v android debug

In [ ]:
# ===== 6. 查看生成的 APK =====
!ls -lh phone_proxy_apk/bin/*.apk 2>/dev/null || echo "编译中或失败，检查上方日志"

In [ ]:
# ===== 7. 下载 APK 到本地 =====
from google.colab import files
import glob
apks = glob.glob('phone_proxy_apk/bin/*.apk')
if apks:
    for apk in apks:
        print(f'下载: {apk}')
        files.download(apk)
else:
    print('未找到 APK，编译失败')

---
### ⚡ 提示
- 首次编译 20~40 分钟（需下载 Android SDK / NDK）
- 后续编译 5~10 分钟
- Colab 空闲 90 分钟会断开，编译期间偶尔点一下页面
- 如需保留 .buildozer 目录加速下次编译，挂载 Google Drive